# Initialization

Every parameter in a network has a value before training starts: a fully
connected layer mapping 512 inputs to 256 outputs brings a $256 \times 512$
weight matrix filled with random numbers. that section explains why the scale of those
numbers decides whether a deep network trains at all, and derives the Xavier
[@Glorot.Bengio.2010] and He [@He.Zhang.Ren.ea.2015] schemes that
keep signal variance stable across layers. This section is the practical
companion: what the library does when you say nothing, how to impose a scheme
of your own on a whole model or a single block, what transformer-era code adds
on top of Xavier and He, and how to write an initializer that no menu provides.

In [ ]:
import math
import jax
from jax import numpy as jnp
from flax import nnx

## Defaults and When to Override Them

You can usually ignore initialization because the default is sensible.

NNX creates parameters when `nnx.Linear` is constructed, using the explicit
input width to determine fan-in. Its default is LeCun normal: the weight comes
from a normal distribution with standard deviation
$1/\sqrt{n_\textrm{in}}$, truncated so that no entry lands in the far tails
(the initializer rescales after clipping to keep the standard deviation on target, which
puts the hard bound near $2.3$ standard deviations), and the bias starts at
zero. Both claims are cheap to check against a fresh layer:

In [ ]:
layer = nnx.Linear(512, 256, rngs=nnx.Rngs(0))
w = layer.kernel
print(f'std {w.std():.4f}, predicted {1 / math.sqrt(512):.4f}')
print(f'max |w| {jnp.abs(w).max():.4f}, '
      f'bias all zero: {bool((layer.bias == 0).all())}')

Other libraries make different but equally fan-aware choices, with one legacy
exception:

| Library | Default for a dense layer's weight |
|:--|:--|
| PyTorch | $U(-1/\sqrt{n_\textrm{in}},\, 1/\sqrt{n_\textrm{in}})$, bias likewise |
| Flax (JAX) | LeCun normal: truncated normal, std $1/\sqrt{n_\textrm{in}}$; zero bias |
| Keras (TensorFlow) | Glorot uniform: $U(\pm\sqrt{6/(n_\textrm{in} + n_\textrm{out})})$; zero bias |
| MXNet Gluon | $U(-0.07, 0.07)$, not fan-aware, so override it; zero bias |

When should you override? Four situations recur: a deep network without
normalization layers, where variance compounds across depth (we demonstrate
this below); reproducing a paper whose results depend on its initialization
recipe; architecture-specific corrections such as the residual scaling later
in this section; and parameters you create by hand, though here NNX forces
the choice on you: construct the value explicitly and wrap it in `nnx.Param`.
NNX layers receive their input and output widths in the constructor, and
their initializers run there as well; no later `init` pass infers fan-in from
the first input.

## Applying Initializers

NNX affine layers such as `Linear` and `Conv` accept `kernel_init` and
`bias_init`, each a function
`(key, shape, dtype) -> array`, and `nnx.initializers` supplies the standard
menu. These functions run in the constructor. For a model that already
exists, we can walk its modules and assign new values to selected parameters.
For a parameter in a custom module, call an initializer directly and wrap its
result in `nnx.Param`. We show the first two routes here.

In [ ]:
init_normal = nnx.initializers.normal(0.01)
net = nnx.Sequential(
    nnx.Linear(4, 8, kernel_init=init_normal,
               bias_init=nnx.initializers.zeros, rngs=nnx.Rngs(0)),
    nnx.relu,
    nnx.Linear(8, 1, kernel_init=init_normal,
               bias_init=nnx.initializers.zeros, rngs=nnx.Rngs(1)))

net.layers[0].kernel[:, 0], net.layers[0].bias[0]

Mixing schemes per block needs no dispatch at all: each layer names its own
initializer, so the choice sits exactly where the layer is declared. Below we
give the first layer Xavier initialization (that section) and set
the last to a constant, a poor idea for training by the symmetry argument of
that section, but it makes the mechanics visible:

In [ ]:
net = nnx.Sequential(
    nnx.Linear(4, 8, kernel_init=nnx.initializers.xavier_uniform(),
               rngs=nnx.Rngs(0)), nnx.relu,
    nnx.Linear(8, 1, kernel_init=nnx.initializers.constant(42.0),
               rngs=nnx.Rngs(1)))

net.layers[0].kernel[:, 0], net.layers[2].kernel

The second route walks the existing object graph. `nnx.iter_modules` yields
each child with its path, and one fresh key per matching layer keeps the draws
independent. The function below re-draws every linear kernel of the
constant-42 network; biases pass through untouched:

In [ ]:
def reinit(model, key, init_fn):
    layers = [m for _, m in nnx.iter_modules(model)
              if isinstance(m, nnx.Linear)]
    for layer, subkey in zip(layers, jax.random.split(key, len(layers))):
        layer.kernel[...] = init_fn(
            subkey, layer.kernel.shape, layer.kernel.dtype)

reinit(net, jax.random.key(1), nnx.initializers.xavier_uniform())
net.layers[2].kernel[:3]

## Modern Schemes: Truncation, Depth, and Zeros

Xavier and He set the variance of a single layer. The schemes below, standard
in transformer codebases, adjust what happens in the distribution's tails,
across depth, and at a block's start.

### Truncated Normals

A Gaussian gets the variance right, but its tails are unbounded. That is
harmless for one draw and a near-certainty at scale: among the $10^8$ weights
of a BERT-sized model, dozens land beyond five standard deviations. Large
draws also consume disproportionate dynamic range once a model is cast to low
precision (that section). BERT and implementations in the ViT
lineage use truncated-normal initialization
[@Devlin.Chang.Lee.ea.2018; @Dosovitskiy.Beyer.Kolesnikov.ea.2021]: the
tails are cut off at a fixed multiple of the nominal standard deviation. Raw
truncated-normal initializers do not necessarily restore the variance removed
with the tails; fan-aware variance-scaling initializers often do.

`nnx.initializers.truncated_normal` states its cutoffs in units of the
standard deviation (`lower=-2.0`, `upper=2.0` by default), so the clip at two
standard deviations is what you get with no extra arguments. Truncation is
the house preference throughout Flax: the `variance_scaling` factory behind
`lecun_normal`, `xavier_normal`, and `he_normal` draws its normal variants
truncated as well.

In [ ]:
key = jax.random.key(0)
w = nnx.initializers.normal(stddev=0.02)(key, (1000, 1000))
print(f'normal:    std {w.std():.4f}, max weight {jnp.abs(w).max():.4f}')
w = nnx.initializers.truncated_normal(stddev=0.02)(key, (1000, 1000))
print(f'truncated: std {w.std():.4f}, max weight {jnp.abs(w).max():.4f}')

A million plain-normal draws produce entries near $5\sigma = 0.1$; the
truncated version guarantees $|w| \leq 0.04$. Its printed standard deviation
dips slightly below the nominal 0.02 because truncation removes tail mass;
practice ignores the difference.

### Scaling Down Residual Branches

A residual network computes $\mathbf{x}_{k+1} = \mathbf{x}_k +
f_k(\mathbf{x}_k)$, so its output is the input plus the sum of $N$ block
contributions. If every block is initialized identically, each contribution
has variance proportional to that of the already-inflated stream it reads, and
the stream's variance compounds geometrically with depth. This is the
depth-wise cousin of the layer-wise explosion in
that section. GPT-2's fix is to shrink only the *last*
linear layer of each residual branch, the output projection that writes into
the stream, by a factor of $1/\sqrt{N}$: with $N$ roughly independent
contributions each scaled down that way, the sum's variance stays $O(1)$
regardless of depth [@Radford.Wu.Child.ea.2019]. For an $L$-layer
transformer the published recipe is a normal with std $0.02/\sqrt{2L}$ on the
residual projections — the number of contributions is $N = 2L$, because each
layer writes to the stream twice, once from attention and once from the MLP.
Unlike Xavier or He, the base 0.02 is not derived from fan-in; it is an
empirical constant that worked at GPT-2's widths and stuck.

the figure draws the two regimes side by side.

![A residual stream with additive block contributions, unscaled (left) versus scaled by 1/sqrt(N) (right). Under the independent-contribution approximation, unscaled variance grows with N and scaling keeps the sum O(1). In an unnormalized stack, each branch reads an already inflated stream, so growth can be faster, as the experiment below shows.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-residual-stream.svg)

### Starting a Block at Zero

Zero-initializing *every* weight is fatal: all units in a layer compute the
same output, receive the same gradient, and stay identical forever
(that section). Zero-initializing just the *last* layer
of a residual block is a different and useful move. The branch then
contributes exactly nothing, each block is the identity map, and the network
starts as a shallow function whose depth switches on gradually during
training. Symmetry is not a problem because the branch's earlier layers keep
their random weights. On the first backward pass, the zeroed projection
receives a nonzero gradient because its input is nonzero, while the earlier
layers receive zero gradient through that projection. After the projection
moves off zero, gradient reaches the whole branch. Keep this scheme distinct
from the previous one: GPT-2 makes every residual projection *small but
nonzero*, whereas zero-init makes one layer *exactly zero* so the block starts
as an exact identity.

### Watching the Variance Compound

Claims about variance at depth are cheap to test. We reuse a compact residual
block (it repeats the one from that section) and stack
it $N$ deep:

In [ ]:
he = nnx.initializers.variance_scaling(2.0, 'fan_in', 'truncated_normal')

class ResidualBlock(nnx.Module):
    def __init__(self, d, out_init=he, rngs=None):
        rngs = nnx.Rngs(0) if rngs is None else rngs
        self.fc1 = nnx.Linear(d, 4 * d, kernel_init=he, rngs=rngs)
        self.fc2 = nnx.Linear(4 * d, d, kernel_init=out_init, rngs=rngs)

    def __call__(self, X):
        return X + self.fc2(nnx.relu(self.fc1(X)))

Every dense layer gets He initialization, appropriate for the ReLU inside
the branch; `variance_scaling(2.0, 'fan_in', 'truncated_normal')` is exactly
what `nnx.initializers.he_normal()` expands to. The block takes its output
projection's initializer as a constructor argument, so each treatment is a
different initializer.
Leaving it alone means He again, scaling by $1/\sqrt{N}$ is a closure over
the depth that wraps the same three-argument signature, and zeroing is
`nnx.initializers.zeros`.

In [ ]:
def scaled_he(num_blocks):
    def init(key, shape, dtype=jnp.float32):
        return he(key, shape, dtype) * num_blocks ** -0.5
    return init

def output_std(num_blocks, out_init):
    rngs = nnx.Rngs(0)
    net = nnx.Sequential(*[
        ResidualBlock(64, out_init=out_init(num_blocks), rngs=rngs)
        for _ in range(num_blocks)])
    X = jax.random.normal(jax.random.key(1), (256, 64))
    return net(X).std().item()

One forward pass on unit-variance inputs per depth and treatment:

In [ ]:
tweaks = {'default': lambda n: he,
          'scaled': scaled_he,
          'zero': lambda n: nnx.initializers.zeros}
print(f'{"N":>3}' + ''.join(f'{name:>10}' for name in tweaks))
for n in (2, 8, 32):
    stds = (output_std(n, tweak) for tweak in tweaks.values())
    print(f'{n:>3}' + ''.join(f'{s:>10.3g}' for s in stds))

The default column multiplies by a roughly constant factor per block in this
unnormalized stack, reaching tens of millions by $N=32$. Such initial
activations make optimization impractical. The scaled column stays near a
small constant over the depths tested: scaling cuts each branch's variance by
a factor of $N$, following the independent-contribution approximation above.
The zero column reproduces the input's standard deviation, since every block
is the identity. This forward-pass test does not prove how an optimizer will
behave, but it catches an unusable initialization before training begins.

## Custom Initializers

Occasionally the menu has nothing you need.

An initializer is just a function `(key, shape, dtype) -> array`, the
signature everything in `nnx.initializers` shares, so writing one is no harder
than using one.

Suppose, to make the point vividly, we want weights distributed as

$$
\begin{aligned}
    w \sim \begin{cases}
        U(5, 10) & \textrm{ with probability } \frac{1}{4} \\
            0    & \textrm{ with probability } \frac{1}{2} \\
        U(-10, -5) & \textrm{ with probability } \frac{1}{4}
    \end{cases}
\end{aligned}
$$

Split the key, draw a magnitude from $U(5, 10)$, and draw a factor from
$\{-1, 0, 0, 1\}$, which zeroes the entry with probability one half and keeps
either sign with probability one quarter. Handing the function to
`kernel_init` makes it official; `init` gives every layer an independent key:

In [ ]:
def my_init(key, shape, dtype=jnp.float32):
    mag_key, sign_key = jax.random.split(key)
    mag = jax.random.uniform(mag_key, shape, dtype, minval=5, maxval=10)
    sign = jax.random.choice(sign_key, jnp.array([-1., 0., 0., 1.], dtype),
                             shape)
    return mag * sign

net = nnx.Sequential(
    nnx.Linear(4, 8, kernel_init=my_init, rngs=nnx.Rngs(0)), nnx.relu,
    nnx.Linear(8, 1, rngs=nnx.Rngs(1)))
net.layers[0].kernel[:, :2]

NNX parameters are mutable variables. Assignment updates the value owned by
the layer while preserving the `nnx.Param` object that optimizers and
checkpoints track. One-off surgery therefore looks like ordinary indexed
assignment:

In [ ]:
net.layers[0].kernel[...] += 1
net.layers[0].kernel[0, 0] = 42
net.layers[0].kernel[0]

There is no ordering caveat: NNX parameters are created in the constructor,
so both graph walks and direct assignment can run immediately.

## Summary

NNX initializes parameters in layer constructors with fan-aware defaults;
override them when depth or a paper's recipe demands it. An initializer is a function
`(key, shape, dtype) -> array`, handed to a layer as `kernel_init` at
construction or run over selected modules in an existing object graph.

On top of Xavier and He,
transformer-era code truncates normal tails to bound outliers, shrinks each
residual output projection by $1/\sqrt{N}$ so the stream's variance stays
$O(1)$ at any depth, and zero-initializes a block's last layer to start it as
the identity.

Anything the menu lacks is a few lines of `jax.random` code behind the same
three-argument signature.

## Exercises

1. Instrument the residual stack: record the standard deviation of the
   activation after every block (run the stack one block at a time, or
   capture per-block activations with the tools of that section)
   for the default and scaled treatments at $N=32$, and plot it against
   depth. Which curve matches the geometric-growth prediction?
1. Zero-initialize *all* layers of every block instead of just the output
   projection. The forward pass still returns the input, but what can the
   network learn? Work out which parameters receive a nonzero gradient, and
   relate the answer to the symmetry-breaking argument of
   that section.
1. Write an initializer that fills each parameter from a dictionary keyed by
   parameter name (walk `net.named_parameters()` in PyTorch, `net.weights`
   in TensorFlow, `nnx.iter_modules(net)` in JAX, `net.collect_params()`
   in MXNet). You have re-invented part of checkpoint loading,
   which that section covers.
1. For a normal distribution truncated at $\pm 2\sigma$: what fraction of
   draws does the clip discard, and by how much does it shrink the standard
   deviation? Verify both numbers against the printed output of the truncation
   demo above.